# Week 10 — Higher-Order Markov Emotion Forecasting

## Objective

Week 9 established three next-emotion forecasting baselines:

- Majority
- Persistence
- First-order Markov

The first-order Markov model did not improve over the majority baseline because `neutral` had the highest transition probability from every source emotion.

This experiment investigates whether a longer history of previous emotional states provides additional predictive information.

### Experiments

- Markov-2: use the previous 2 emotions to predict the next emotion.
- Markov-3: use the previous 3 emotions to predict the next emotion.

### Forecasting task

Given:

\[
E_{t-k+1}, \ldots, E_t
\]

predict:

\[
E_{t+1}
\]

Only valid consecutive utterances within the same dialogue are used.

### Primary metric

**Macro F1**

Accuracy and Weighted F1 are reported as secondary metrics.

In [2]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_meld
from src.analysis.forecasting import (
    build_emotion_sequences,
    fit_higher_order_markov_model,
    higher_order_markov_forecast,
)
from src.analysis.trajectories import EMOTIONS

print("Project root:", PROJECT_ROOT)
print("Emotions:", EMOTIONS)

Project root: /home/chitta/Projects/emotion-dynamics-nlp/notebooks
Emotions: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']


In [3]:
from src.config import RAW_DATA_DIR

print("Raw data directory:", RAW_DATA_DIR)

meld = load_meld(RAW_DATA_DIR / "meld")

train_df = meld.train
dev_df = meld.dev
test_df = meld.test

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

Raw data directory: /home/chitta/Projects/emotion-dynamics-nlp/data/raw
Train: (9989, 11)
Dev: (1109, 11)
Test: (2610, 11)


In [4]:
# Build higher-order forecasting examples
train_seq_2 = build_emotion_sequences(train_df, history_size=2)
dev_seq_2 = build_emotion_sequences(dev_df, history_size=2)
test_seq_2 = build_emotion_sequences(test_df, history_size=2)

train_seq_3 = build_emotion_sequences(train_df, history_size=3)
dev_seq_3 = build_emotion_sequences(dev_df, history_size=3)
test_seq_3 = build_emotion_sequences(test_df, history_size=3)

print("Markov-2 examples:")
print("Train:", len(train_seq_2))
print("Dev:  ", len(dev_seq_2))
print("Test: ", len(test_seq_2))

print("\nMarkov-3 examples:")
print("Train:", len(train_seq_3))
print("Dev:  ", len(dev_seq_3))
print("Test: ", len(test_seq_3))

Markov-2 examples:
Train: 7860
Dev:   884
Test:  2059

Markov-3 examples:
Train: 6913
Dev:   779
Test:  1807


In [5]:
# Fit higher-order Markov models using TRAINING examples only

markov2_model = fit_higher_order_markov_model(
    train_seq_2,
    history_size=2,
)

markov3_model = fit_higher_order_markov_model(
    train_seq_3,
    history_size=3,
)

print("Markov-2 histories:", len(markov2_model))
print("Markov-3 histories:", len(markov3_model))

Markov-2 histories: 49
Markov-3 histories: 311


In [6]:
# Generate predictions on the test set

test_pred_2 = higher_order_markov_forecast(
    test_seq_2,
    markov2_model,
    history_size=2,
)

test_pred_3 = higher_order_markov_forecast(
    test_seq_3,
    markov3_model,
    history_size=3,
)

y_test_2 = test_seq_2["next_emotion"].tolist()
y_test_3 = test_seq_3["next_emotion"].tolist()

print("Markov-2 predictions:", len(test_pred_2))
print("Markov-3 predictions:", len(test_pred_3))

Markov-2 predictions: 2059
Markov-3 predictions: 1807


In [7]:
def evaluate_forecast(y_true, y_pred, model_name):
    print(f"\n{'=' * 50}")
    print(model_name)
    print(f"{'=' * 50}")

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=EMOTIONS,
        average="macro",
        zero_division=0,
    )
    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=EMOTIONS,
        average="weighted",
        zero_division=0,
    )

    print(f"Accuracy:   {accuracy:.4f}")
    print(f"Macro F1:    {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")

    print("\nPer-class report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=EMOTIONS,
            target_names=EMOTIONS,
            zero_division=0,
        )
    )

    return {
        "model": model_name,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


results_2 = evaluate_forecast(
    y_test_2,
    test_pred_2,
    "Markov-2",
)

results_3 = evaluate_forecast(
    y_test_3,
    test_pred_3,
    "Markov-3",
)


Markov-2
Accuracy:   0.4881
Macro F1:    0.2003
Weighted F1: 0.3992

Per-class report:
              precision    recall  f1-score   support

       anger       0.46      0.33      0.39       276
     disgust       0.00      0.00      0.00        55
        fear       0.00      0.00      0.00        40
         joy       0.40      0.13      0.20       312
     neutral       0.50      0.87      0.64       974
     sadness       0.35      0.12      0.18       170
    surprise       0.00      0.00      0.00       232

    accuracy                           0.49      2059
   macro avg       0.24      0.21      0.20      2059
weighted avg       0.39      0.49      0.40      2059


Markov-3
Accuracy:   0.4765
Macro F1:    0.2080
Weighted F1: 0.3984

Per-class report:
              precision    recall  f1-score   support

       anger       0.40      0.31      0.35       247
     disgust       0.00      0.00      0.00        49
        fear       0.00      0.00      0.00        38
         j

In [8]:
import inspect

print(inspect.signature(higher_order_markov_forecast))
print(inspect.getsource(higher_order_markov_forecast))

(sequence_examples: 'pd.DataFrame', transition_model: 'dict[tuple[str, ...], pd.Series]', history_size: 'int', fallback: 'str' = 'neutral') -> 'list[str]'
def higher_order_markov_forecast(
    sequence_examples: pd.DataFrame,
    transition_model: dict[tuple[str, ...], pd.Series],
    history_size: int,
    fallback: str = "neutral",
) -> list[str]:
    """Forecast next emotion using a higher-order Markov model."""
    if history_size < 1:
        raise ValueError("history_size must be at least 1.")
    if fallback not in EMOTIONS:
        raise ValueError(f"Invalid fallback emotion: {fallback}")

    history_columns = [f"history_{j + 1}" for j in range(history_size)]
    missing = set(history_columns) - set(sequence_examples.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    predictions = []
    for _, row in sequence_examples.iterrows():
        history = tuple(str(row[column]) for column in history_columns)
        predictions.app

In [9]:
def recursive_forecast(
    df,
    transition_model,
    history_size,
    fallback="neutral",
):
    """
    Recursively forecast the next emotion within each dialogue.

    The first `history_size` emotions are ground truth.
    After that, the model's predictions are fed back as history.
    Only consecutive utterance IDs are used.
    """
    required = {"dialogue_id", "utterance_id", "emotion"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    data = df.copy().sort_values(
        ["dialogue_id", "utterance_id"]
    ).reset_index(drop=True)

    y_true = []
    y_pred = []

    for _, dialogue in data.groupby("dialogue_id", sort=False):
        dialogue = dialogue.reset_index(drop=True)

        emotions = dialogue["emotion"].astype(str).tolist()
        utterance_ids = dialogue["utterance_id"].tolist()

        # Find runs of consecutive utterance IDs.
        run_start = 0

        for i in range(1, len(dialogue) + 1):
            is_end = (
                i == len(dialogue)
                or utterance_ids[i] != utterance_ids[i - 1] + 1
            )

            if is_end:
                run = emotions[run_start:i]

                if len(run) > history_size:
                    # Start with ground-truth history.
                    history = list(run[:history_size])

                    for target in run[history_size:]:
                        history_tuple = tuple(history[-history_size:])

                        if history_tuple in transition_model:
                            prediction = transition_model[
                                history_tuple
                            ].idxmax()
                        else:
                            prediction = fallback

                        y_true.append(target)
                        y_pred.append(prediction)

                        # Feed prediction back into the history.
                        history.append(prediction)

                run_start = i

    return y_true, y_pred


print("Recursive forecasting helper ready.")

Recursive forecasting helper ready.


In [10]:
# Recursive forecasting: model predictions are fed back as history

recursive_true_2, recursive_pred_2 = recursive_forecast(
    test_df,
    markov2_model,
    history_size=2,
)

recursive_true_3, recursive_pred_3 = recursive_forecast(
    test_df,
    markov3_model,
    history_size=3,
)

print("Markov-2 recursive examples:", len(recursive_true_2))
print("Markov-3 recursive examples:", len(recursive_true_3))

Markov-2 recursive examples: 2059
Markov-3 recursive examples: 1807


In [11]:
recursive_results_2 = evaluate_forecast(
    recursive_true_2,
    recursive_pred_2,
    "Markov-2 — Recursive",
)

recursive_results_3 = evaluate_forecast(
    recursive_true_3,
    recursive_pred_3,
    "Markov-3 — Recursive",
)


Markov-2 — Recursive
Accuracy:   0.4424
Macro F1:    0.1516
Weighted F1: 0.3488

Per-class report:
              precision    recall  f1-score   support

       anger       0.26      0.17      0.21       276
     disgust       0.00      0.00      0.00        55
        fear       0.00      0.00      0.00        40
         joy       0.27      0.13      0.17       312
     neutral       0.48      0.84      0.61       974
     sadness       0.21      0.04      0.07       170
    surprise       0.00      0.00      0.00       232

    accuracy                           0.44      2059
   macro avg       0.18      0.17      0.15      2059
weighted avg       0.32      0.44      0.35      2059


Markov-3 — Recursive
Accuracy:   0.4438
Macro F1:    0.1345
Weighted F1: 0.3333

Per-class report:
              precision    recall  f1-score   support

       anger       0.24      0.13      0.17       247
     disgust       0.00      0.00      0.00        49
        fear       0.00      0.00      0

In [12]:
import numpy as np

# Compare fixed-history predictions against recursive predictions
# on the same valid test examples.

divergence_2 = np.mean(
    np.array(test_pred_2) != np.array(recursive_pred_2)
)

divergence_3 = np.mean(
    np.array(test_pred_3) != np.array(recursive_pred_3)
)

print(f"Markov-2 prediction divergence: {divergence_2:.4f} ({divergence_2 * 100:.2f}%)")
print(f"Markov-3 prediction divergence: {divergence_3:.4f} ({divergence_3 * 100:.2f}%)")

Markov-2 prediction divergence: 0.2414 (24.14%)
Markov-3 prediction divergence: 0.2452 (24.52%)


In [13]:
week10_results = pd.DataFrame([
    {
        "Model": "Majority",
        "Accuracy": 0.4794,
        "Macro F1": 0.0926,
        "Weighted F1": 0.3107,
    },
    {
        "Model": "Persistence",
        "Accuracy": 0.4192,
        "Macro F1": 0.2736,
        "Weighted F1": 0.4186,
    },
    {
        "Model": "Markov-1",
        "Accuracy": 0.4794,
        "Macro F1": 0.0926,
        "Weighted F1": 0.3107,
    },
    {
        "Model": "Markov-2 (Ground Truth)",
        "Accuracy": 0.4881,
        "Macro F1": 0.2003,
        "Weighted F1": 0.3992,
    },
    {
        "Model": "Markov-3 (Ground Truth)",
        "Accuracy": 0.4765,
        "Macro F1": 0.2080,
        "Weighted F1": 0.3984,
    },
    {
        "Model": "Markov-2 (Recursive)",
        "Accuracy": 0.4424,
        "Macro F1": 0.1516,
        "Weighted F1": 0.3488,
    },
    {
        "Model": "Markov-3 (Recursive)",
        "Accuracy": 0.4438,
        "Macro F1": 0.1345,
        "Weighted F1": 0.3333,
    },
])

week10_results

,Model,Accuracy,Macro F1,Weighted F1
0,Majority,0.4794,0.0926,0.3107
1,Persistence,0.4192,0.2736,0.4186
2,Markov-1,0.4794,0.0926,0.3107
3,Markov-2 (Ground Truth),0.4881,0.2003,0.3992
4,Markov-3 (Ground Truth),0.4765,0.2080,0.3984
5,Markov-2 (Recursive),0.4424,0.1516,0.3488
6,Markov-3 (Recursive),0.4438,0.1345,0.3333


In [14]:
error_propagation = pd.DataFrame([
    {
        "Model": "Markov-2",
        "Fixed Macro F1": 0.2003,
        "Recursive Macro F1": 0.1516,
        "Macro F1 Drop": 0.2003 - 0.1516,
        "Prediction Divergence": divergence_2,
    },
    {
        "Model": "Markov-3",
        "Fixed Macro F1": 0.2080,
        "Recursive Macro F1": 0.1345,
        "Macro F1 Drop": 0.2080 - 0.1345,
        "Prediction Divergence": divergence_3,
    },
])

error_propagation

,Model,Fixed Macro F1,Recursive Macro F1,Macro F1 Drop,Prediction Divergence
0,Markov-2,0.2003,0.1516,0.0487,0.241379
1,Markov-3,0.2080,0.1345,0.0735,0.245158
